# Entrenar una xarxa neuronal: experimentem amb MNIST

**Objectiu:** entendre què passa *mentre* una xarxa s'entrena i com els hiperparàmetres (learning rate, batch i nombre d'èpoques) canvien el resultat.

**Com treballarem:** a cada experiment
1. **Prediu** per escrit què passarà (cel·la de resposta).
2. **Executa** l'experiment.
3. **Compara** i explica per què la teva predicció era correcta o no.

Les preguntes s'han de contestar al formulari. Estan indicades amb 🟪 o 🟦. Les preguntes marcades en lila s'han de respondre abans d'executar. 

Xarxa que farem servir a tota la pràctica: imatge 28×28 = 784 píxels → una capa oculta de 64 neurones (ReLU) → 10 sortides (dígits 0–9).

## 0. Preparació (només cal executar-ho)

Dividim les dades en dues parts que **no es barregen mai**:
- **Entrenament**: la xarxa les fa servir per ajustar els pesos.
- **Validació**: la xarxa *no* aprèn d'elles; només les fem servir per veure com se'n surt amb dades que no ha vist.

In [ ]:
%pip install torch torchvision matplotlib

import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets

mnist = datasets.MNIST("data", train=True, download=True)
X = (mnist.data.float() / 255).flatten(1)   # cada imatge és un vector de 784 valors entre 0 i 1
y = mnist.targets

g = torch.Generator().manual_seed(0)
p = torch.randperm(len(X), generator=g)
X_val, y_val = X[p[:10000]], y[p[:10000]]       # validació: 10 000 imatges
X_pool, y_pool = X[p[10000:]], y[p[10000:]]     # imatges disponibles per entrenar
print("Entrenament disponible:", len(X_pool), "| Validació:", len(X_val))

fig, axs = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axs, X_pool, y_pool):
    ax.imshow(img.view(28, 28), cmap="gray"); ax.set_title(int(lab)); ax.axis("off")

## 1. La funció d'entrenament

Llegeix el codi: és el cor de tot el que farem. La xarxa utilitza sempre la mateixa arquitectura: `784 → 64 → 10`, amb una capa oculta de 64 neurones i activació ReLU.

Cada cop que la xarxa rep un *batch* (un grup d'imatges), fa aquests quatre passos per aprendre d'aquells exemples:

1. **Forward**: la xarxa fa una predicció per a cada imatge del batch.
2. **Loss**: comparem les prediccions amb les respostes correctes i mesurem l'error (*cross-entropy*: com més gran, pitjor).
3. **Backward**: backpropagation calcula com caldria canviar els pesos per reduir aquest error.
4. **Step**: s'actualitzen els pesos segons aquests càlculs: `pes = pes - lr * gradient`.

Després, el procés es repeteix amb el batch següent.

In [ ]:
def entrena(lr=0.1, batch=64, epoques=10, n_train=10000, seed=0):
    torch.manual_seed(seed)
    ordre_generator = torch.Generator().manual_seed(seed + 1)
    model = nn.Sequential(nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    Xt, yt = X_pool[:n_train], y_pool[:n_train]
    h = {"train": [], "val": [], "acc_val": [], "actualitzacions": 0}

    for ep in range(epoques):                      # UNA ÈPOCA = recórrer TOTES les dades d'entrenament una vegada
        ordre = torch.randperm(n_train, generator=ordre_generator)
        for i in range(0, n_train, batch):         # UN BATCH = un grup petit d'exemples
            idx = ordre[i:i + batch]
            loss = F.cross_entropy(model(Xt[idx]), yt[idx])   # 1 forward + 2 loss
            opt.zero_grad(); loss.backward()                  # 3 backward
            opt.step()                                        # 4 UNA ACTUALITZACIÓ dels pesos
            h["actualitzacions"] += 1

        with torch.no_grad():                      # al final de cada època mesurem (sense aprendre)
            h["train"].append(F.cross_entropy(model(Xt), yt).item())
            sortida = model(X_val)
            h["val"].append(F.cross_entropy(sortida, y_val).item())
            h["acc_val"].append((sortida.argmax(1) == y_val).float().mean().item())
    return h


def grafica(resultats):
    """resultats: diccionari {nom: historial}. Contínua = train, discontínua = validació."""
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for nom, h in resultats.items():
        l, = axs[0].plot(h["train"], label=nom + " (train)")
        axs[0].plot(h["val"], "--", color=l.get_color(), label=nom + " (val)")
        axs[1].plot(h["acc_val"], color=l.get_color(), label=nom)
        print(f"{nom:>22} | loss train {h['train'][-1]:.3f} | loss val {h['val'][-1]:.3f} | "
              f"encert val {h['acc_val'][-1]:.1%} | actualitzacions {h['actualitzacions']}")
    axs[0].set(title="Loss", xlabel="època"); axs[0].legend(fontsize=7)
    axs[1].set(title="Encert en validació", xlabel="època"); axs[1].legend(fontsize=7)
    plt.show()

## 2. Època, batch i actualització de pesos

Aquests tres conceptes es confonen sovint. Fixa't en el codi anterior:

- **Batch**: grup d'exemples que es processen junts.
- **Actualització**: un cop de `opt.step()`; passa **una vegada per batch**.
- **Època**: quan s'han vist totes les dades d'entrenament una vegada.

### Cas 2.1
Entrenem amb `n_train=10000` imatges, `batch=100` i `epoques=3`.

**🟪 PR1:** Quants batches hi ha en una època? 

**🟪 PR2:** Quantes actualitzacions de pesos en total? 

In [ ]:
h = entrena(batch=100, epoques=3, n_train=10000)
print("Actualitzacions reals:", h["actualitzacions"])

### Cas 2.2
**🟪 PR3:** Ara fem `batch=1000` (mateixes 3 èpoques, mateixes dades). Quantes actualitzacions hi haurà? 

**🟪 PR4:** Amb menys actualitzacions, creus que la xarxa haurà après més o menys en 3 èpoques?

In [ ]:
grafica({"batch=100": entrena(batch=100, epoques=3),
         "batch=1000": entrena(batch=1000, epoques=3)})

**🟦 PR5**: Completa amb les teves paraules: «*Amb el mateix nombre d’èpoques, un batch més petit significa ______ actualitzacions, i per tant ______ oportunitats perquè els pesos s’ajustin i la xarxa aprengui.*» 

## 3. Primer entrenament i lectura de les corbes

- **Loss d'entrenament** (línia contínua): com d'equivocada està la xarxa amb les dades que *està fent servir per aprendre*.
- **Loss de validació** (línia discontínua): com d'equivocada està amb dades que *mai ha vist*. És la que realment ens importa, perquè mesura si la xarxa **generalitza**.

### Cas 3.1
Entrenem 20 èpoques amb `lr=0.1`. 

**🟪 PR6:** Dibuixa com creus que evolucionaran els valors de la loss i de l'accuracy. Quines diferències trobarem en els valors de loss d'entrenament i de validació. Hauran de ser iguals?

In [ ]:
base = entrena(lr=0.1, epoques=20)
grafica({"base lr=0.1": base})

**Preguntes:**

**🟦 PR7:** La loss de train baixa sempre? I la de validació? Quina de les dues és més baixa? Per què creus que és així? 

**🟦 PR8:** Per què l'encert en validació puja ràpidament al principi i després quasi no millora?


## 4. El learning rate

El learning rate (`lr`) és la mida del pas que fa cada pes a cada actualització: `pes = pes - lr * gradient`.

Pensa en baixar una vall amb els ulls tancats: passos massa petits tarden molt; passos massa grans et fan saltar d'una paret a l'altra de la vall.

### Cas 4.1
Provarem `lr = 0.001`, `0.1` i `5`. 

**🟪 PR9:** Per a cadascun, escriu què esperes veure (loss i encert final aproximats, forma de la corba):

| lr | Predicció |
|---|---|
| 0.001 | |
| 0.1 | |
| 5 | |

In [ ]:
grafica({f"lr={lr}": entrena(lr=lr, epoques=20) for lr in [0.001, 0.1, 5]})

**Preguntes:**

**🟦 PR10:** Quin lr és **massa lent**? Quina evidència hi ha al gràfic? (Pista: la loss encara baixa quan acaba l'entrenament.)

**🟦 PR11:** Quin és **inestable**? Quina forma tenen les corbes? (Pista: saltirons, pujades, valors que no baixen.)

**🟦 PR12:** Si un lr massa petit "tampoc no falla", per què no triem sempre el més petit possible?


### Cas 4.2
**🟦 PR13:** Troba un lr **entre 0.1 i 5** on la xarxa comenci a anar malament. Prova diversos valors (p. ex. `[0.1, 0.8, 2]`) i anota a partir d'on es nota inestabilitat.

**🟦 PR14:** Quan una xarxa és massa lenta, es pot compensar amb **més èpoques**. Prova amb `lr=0.001` quantes èpoques necessitem per arribar allà mateix que `lr=0.1`?

In [ ]:
# Escriu aquí la teva llista de lr a provar
grafica({f"lr={lr}": entrena(lr=lr, epoques=20) for lr in [0.1, 0.8, 2]})

## 5. Sobreajustament (overfitting)

Si la xarxa té prou capacitat i poques dades, en comptes d'aprendre **patrons** pot **memoritzar** els exemples concrets d'entrenament.

Símptoma típic: la loss de train continua baixant, però la de validació **s'atura i després puja**.

### Cas 5.1
Entrenarem amb només `n_train=300` imatges i 200 èpoques. Comparem amb la mateixa xarxa, entrenada amb 10 000 imatges.

**🟪 PR15:** Quina de les dues tindrà més diferència entre loss de train i de validació?

**🟪 PR16:** La loss de train de la xarxa de 300 imatges, arribarà a ser molt baixa? 

In [ ]:
grafica({"300 imatges": entrena(n_train=300, epoques=200, batch=32),
         "10000 imatges": entrena(n_train=10000, epoques=200, batch=32)})

**Preguntes:**

**🟦 PR17:** A partir de quina època comença a notar-se el sobreajustament a la xarxa de 300 imatges? (Mira on la línia discontínua deixa de baixar.)

**🟦 PR18:** Si només miréssim la loss de train, quina conclusió errònia en trauríem?

**🟦 PR19:** Digues dues maneres de reduir el sobreajustament amb el que has vist avui (pista: dades, èpoques, mida de la xarxa).

## 6. Diagnòstic: quin problema té cada entrenament?

Sense mirar el codi, mira aquests tres entrenaments. A cada gràfic, escriu si el símptoma és **massa lent**, **inestable**, **sobreajustament** o **correcte**, i justifica amb una observació concreta de les corbes.

In [ ]:
misteri = {"A": entrena(lr=0.003, epoques=20),
           "B": entrena(lr=3, epoques=20),
           "C": entrena(lr=0.1, n_train=300, epoques=150, batch=32)}
grafica(misteri)

**🟦 PR20**

| Entrenament | Diagnòstic |
|---|---|
| A | |
| B | |
| C | |

## 7. Repte final: tria i justifica la teva configuració

Fes **com a mínim 4 proves** variant `lr`, `batch` i/o `epoques` (amb `n_train=10000`). L'arquitectura es manté fixa en una capa oculta de 64 neurones. Objectiu: **la millor loss de validació possible sense sobreajustar ni inestabilitat**.

Regla: canvia **una sola cosa cada vegada** respecte la prova anterior, o no sabràs què ha causat el canvi.

In [ ]:
proves = {
    "prova 1: base": dict(lr=0.1, batch=64, epoques=20),
    "prova 2: lr menor": dict(lr=0.05, batch=64, epoques=20),
    "prova 3: batch més gran": dict(lr=0.05, batch=128, epoques=20),
    "prova 4: més èpoques": dict(lr=0.05, batch=128, epoques=30),
}
grafica({nom: entrena(**cfg) for nom, cfg in proves.items()})

### **🟦 PR21:** Justificació

Respon amb **dades concretes** (números de loss/encert o forma de les corbes). Frases com "funciona millor" no són vàlides.


1. **Configuració triada** (lr, batch, èpoques): 
2. **Loss de validació i encert que obté**: 
3. **Per què aquest lr?** Compara amb un lr més gran i un de més petit que hagis provat: 
4. **Per què aquest nombre d'èpoques?** Què passa si en poses menys o més?: 
5. **Hi ha senyals de sobreajustament?** Quina diferència hi ha entre loss de train i de validació?: 
6. **Puja una imatge del gràfic** que justifiqui la teva elecció.